# Project 2 OpenCV Experiments

Upload one photo of your tape track to Colab and name it `test_frame.jpg`. Use a photo taken from roughly the same forward-down camera angle you will use on the robot.

This notebook lets you calibrate the lane-detection pipeline on a still image before running the robot.

<div style='border-left: 8px solid #d29922; background: #fff7d6; color: #1f2328; padding: 18px; border-radius: 8px; margin: 16px 0;'>
<div style='font-weight: 800; color: #7c2d12; letter-spacing: 0.08em;'>WARNING</div>
<h2 style='margin-top: 4px;'>How to take the right test photo</h2>
<p>The row scanning algorithm expects the tape to look like it does from the robot's camera: a stripe running away from you into the distance, narrow at the top and slightly wider at the bottom.</p>
<p><strong>CORRECT photo:</strong> stand behind the robot position and hold your phone at roughly a 50-60 degree angle pointing forward-down at the tape. The tape should appear as a vertical stripe running from the bottom of the frame toward the top, like a road disappearing into the distance.</p>
<p><strong>WRONG photo:</strong> holding the phone straight down above the tape. This produces a horizontal blob in the center of the frame. Row scanning will miss it because it scans horizontal rows looking for a vertical stripe, and because the blob sits in the middle rows instead of the bottom rows where the scan lines are.</p>
<div style='display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 12px; margin: 14px 0;'>
<div style='border: 1px solid #d0b66b; border-radius: 8px; overflow: hidden; background: #fff;'><div style='padding: 8px 10px; font-weight: 700;'>WRONG: straight down</div><div style='height: 190px; position: relative; background: #e8e2d7;'><div style='position: absolute; left: 18%; right: 18%; top: 47%; height: 28px; border-radius: 999px; background: #111827; transform: rotate(3deg);'></div><div style='position: absolute; left: 8%; right: 8%; top: 60%; height: 2px; background: #facc15;'></div><div style='position: absolute; left: 8%; right: 8%; top: 75%; height: 2px; background: #facc15;'></div><div style='position: absolute; left: 8%; right: 8%; top: 90%; height: 2px; background: #facc15;'></div></div><div style='padding: 8px 10px;'>Horizontal blob in the center rows.</div></div>
<div style='border: 1px solid #d0b66b; border-radius: 8px; overflow: hidden; background: #fff;'><div style='padding: 8px 10px; font-weight: 700;'>CORRECT: robot camera angle</div><div style='height: 190px; position: relative; overflow: hidden; background: #e8e2d7;'><div style='position: absolute; left: 36%; bottom: -10%; width: 28%; height: 112%; clip-path: polygon(44% 0, 56% 0, 82% 100%, 18% 100%); background: #111827;'></div><div style='position: absolute; left: 8%; right: 8%; top: 60%; height: 2px; background: #facc15;'></div><div style='position: absolute; left: 8%; right: 8%; top: 75%; height: 2px; background: #facc15;'></div><div style='position: absolute; left: 8%; right: 8%; top: 90%; height: 2px; background: #facc15;'></div></div><div style='padding: 8px 10px;'>Vertical stripe reaches the lower scan rows.</div></div>
</div>
<p>If you only have a horizontal photo right now, you can rotate it in the notebook to simulate the correct perspective. Add this line after <code>cv2.imread</code>:</p>
<pre><code>frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)</code></pre>
<p>This is the same rotation the robot code applies in <code>main()</code>. Run it and the horizontal tape becomes a vertical stripe.</p>
</div>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

BINARY_THRESHOLD = 70  # Starting value for the quick check. Tune it in Cell 4.

# Load your photo.
frame = cv2.imread("test_frame.jpg")
if frame is None:
    raise FileNotFoundError("Upload your track photo and name it test_frame.jpg")

# If your photo is horizontal, uncomment this line before continuing.
# frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

print(f"Image size: {frame.shape[1]} x {frame.shape[0]} pixels")

# OpenCV uses BGR order. Matplotlib expects RGB.
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title("Your camera frame")
plt.axis("off")
plt.show()

## Cell 1b: Quick row-location sanity check

Run this before any other processing. It shows which horizontal rows contain likely tape pixels so you can tell whether the photo angle is usable before row scanning.

In [ ]:
# Quick check — where is the tape in your image?
gray_check = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
_, binary_check = cv2.threshold(gray_check, BINARY_THRESHOLD,
                                255, cv2.THRESH_BINARY_INV)

print("Rows containing white pixels (potential tape rows):")
for ry in range(0, frame.shape[0], int(frame.shape[0]/20)):
    count = np.count_nonzero(binary_check[ry])
    bar = "█" * min(count // 10, 40)
    print(f"  {ry/frame.shape[0]:.0%} from top: {bar} ({count}px)")

print()
print("The tape should appear as a vertical band of rows")
print("concentrated in the LOWER 60% of the image (60-100% from top)")
print("If it appears in the middle rows only, rotate your photo first.")

## Cell 2: Convert to grayscale

The lane detector only needs brightness. Grayscale turns each pixel into one value from 0 to 255.

In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(frame_rgb)
plt.title("Original color")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Color image shape: {frame.shape}")
print(f"Grayscale shape:   {gray.shape}")

## Cell 3: Gaussian blur

Blur removes tiny sensor noise before thresholding. Try `(3, 3)` for mild blur or `(15, 15)` for heavy blur.

In [ ]:
blur = cv2.GaussianBlur(gray, (7, 7), 0)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Before blur")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(blur, cmap="gray")
plt.title("After 7x7 Gaussian blur")
plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 4: Binary threshold

This is the key calibration cell. Change `BINARY_THRESHOLD` and run the cell again until only the tape is white and the floor is black.

Write down the value that works. That value goes into `lane_follower_adv.py`.

In [ ]:
BINARY_THRESHOLD = 70  # Change this value and run again.

_, binary = cv2.threshold(blur, BINARY_THRESHOLD, 255, cv2.THRESH_BINARY_INV)

plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(blur, cmap="gray")
plt.title("Blurred")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(binary, cmap="gray")
plt.title(f"Binary threshold={BINARY_THRESHOLD}\nwhite = tape, black = floor")
plt.axis("off")
plt.tight_layout()
plt.show()

white_pixels = np.count_nonzero(binary)
total_pixels = binary.size
print(f"White pixels: {white_pixels} ({100 * white_pixels / total_pixels:.1f}% of image)")
print("These are the pixels the algorithm thinks are tape")

Too low, such as `40`: too few pixels become white and parts of the tape may disappear.

Too high, such as `120`: too many pixels become white and the floor may get counted as tape.

`cv2.THRESH_BINARY_INV` is used because dark tape should become white in the binary image. For white tape on a dark floor, use `cv2.THRESH_BINARY` in the robot code instead.

## Cell 5: Morphological cleaning

`OPEN` removes small white specks. `CLOSE` fills small gaps in the white tape region.

In [ ]:
kernel = np.ones((7, 7), np.uint8)
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(
    axes,
    [binary, opened, clean],
    ["After threshold", "After OPEN: noise removed", "After CLOSE: gaps filled"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

Try changing the kernel to `(3, 3)`, `(9, 9)`, or `(11, 11)`. A larger kernel removes more noise but can damage a thin tape line.

## Cell 6: Row scanning

This finds the center of the tape on several horizontal rows. The magenta dots should sit on the tape.

Important: this notebook does **not** apply ROI cropping before row scanning. The full image is processed. The default `ROW_FRACS` values scan the lower portion of the full image, which matches a forward-down robot camera view. If your test photo was taken straight down or the tape appears in the upper or middle half of the image, these rows can miss the tape entirely.

In [ ]:
h, w = clean.shape
ROW_FRACS = [0.90, 0.75, 0.60, 0.45, 0.30]

print(f"clean shape: {clean.shape}")
print(f"clean dtype: {clean.dtype}")
print(f"Total white pixels in clean: {np.count_nonzero(clean)}")
print(f"Scan rows (pixel y): {[int(clean.shape[0]*f) for f in ROW_FRACS]}")

display = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)
points = []

for frac in ROW_FRACS:
    ry = int(h * frac)
    xs = np.where(clean[ry] > 0)[0]

    cv2.line(display, (0, ry), (w, ry), (255, 150, 0), 1)

    if len(xs) < 15:
        print(f"Row {frac:.0%}: only {len(xs)} white pixels, skipped")
        continue

    runs = np.split(xs, np.where(np.diff(xs) > 1)[0] + 1)
    best = max(runs, key=len)

    if len(best) < 12:
        print(f"Row {frac:.0%}: longest run only {len(best)} pixels, skipped")
        continue

    cx = int((best[0] + best[-1]) / 2)
    points.append((cx, ry))

    cv2.circle(display, (cx, ry), 8, (255, 0, 255), -1)
    print(f"Row {frac:.0%}: tape center at x={cx} (run width: {len(best)} px)")

if len(points) >= 2:
    cv2.line(display, points[0], points[-1], (0, 100, 255), 2)
    print(f"\nNear point: {points[0]}")
    print(f"Far point:  {points[-1]}")
    print(f"Horizontal shift: {points[-1][0] - points[0][0]} px")

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
plt.title("Row scanning: blue=scan lines, magenta=centers, orange=direction")
plt.axis("off")
plt.show()

If the dots are wrong, go back to Cell 4 and tune `BINARY_THRESHOLD`. If no dots appear even though the tape is visible, reduce the 15-pixel and 12-pixel minimums.

## Cell 7: Two-point steering error

The robot computes this same number every frame while it is moving.

In [ ]:
if len(points) >= 2:
    near_cx = points[0][0]
    far_cx = points[-1][0]

    position_error = (near_cx - w / 2) / w
    direction_error = (far_cx - near_cx) / w
    raw_angle_error = 0.25 * position_error + 0.75 * direction_error

    print("=" * 50)
    print(f"Frame center:    x = {w // 2}")
    print(f"Near point:      x = {near_cx}")
    print(f"Far point:       x = {far_cx}")
    print()
    print(f"Position error:  {position_error:+.3f}")
    print(f"Direction error: {direction_error:+.3f}")
    print(f"Combined error:  {raw_angle_error:+.3f}")
    print()
    if raw_angle_error > 0.05:
        print("Robot should steer RIGHT")
    elif raw_angle_error < -0.05:
        print("Robot should steer LEFT")
    else:
        print("Line is approximately centered, robot drives straight")
else:
    print("Not enough points detected to compute steering error")
    print("Check your BINARY_THRESHOLD value")

## Values to take back to the robot code

- `BINARY_THRESHOLD = _____`
- Kernel size working well: `_____ x _____`
- Typical points count on a straight line: `_____`

Put these values into the configuration section of `lane_follower_adv.py` before starting the robot.